# BNPL — Merchant Industry Segmentation and Top 10 Rankings

**Objective:** Create five interpretable industry segments, then recommend the **Top 10 merchants per segment** using the *existing overall ranking score* (including predicted 3-month revenue and fraud penalty). This notebook does **not** change the Top 100 overall ranking.

**Method:** Parse messy merchant tags; inspect industries; apply transparent keyword mapping; rank within each segment; audit coverage, risk flags and outputs. Merchants without an interpretable industry remain **Unknown** and unmatched industries remain **Other** (both retained in full outputs but excluded from the five segment Top 10 lists).

## 1. Load overall merchant rankings

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'curated').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate data/curated. Open this notebook from inside the project repository.')

INPUT = ROOT / 'data' / 'curated' / 'merchant_rankings' / 'merchant_rankings_all.parquet'
OUTPUT = ROOT / 'data' / 'curated' / 'merchant_segments'
OUTPUT.mkdir(parents=True, exist_ok=True)

merchants = pd.read_parquet(INPUT).copy()
required = {'merchant_abn', 'ranking_score', 'tags'}
missing = required - set(merchants.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
if merchants['merchant_abn'].duplicated().any():
    raise ValueError('Expected exactly one row per merchant_abn.')
if merchants['ranking_score'].isna().any():
    raise ValueError('Ranking scores contain missing values; rerun merchant_ranking.ipynb.')
print(f'Loaded {len(merchants):,} merchants from {INPUT}')
print('Available columns:', merchants.columns.tolist())

## 2. Parse industry, tier and take rate from messy `tags`

The tag strings vary in brackets, case and punctuation. The parser anchors on the **tier immediately before `take rate:`** rather than splitting on commas (which also occur inside industry names). Take rates are extracted for descriptive analysis only; they are **not** added to the established ranking score.

In [ ]:
# Example: '([digital goods: books, movies, music], [a], [take rate: 6.99])'
# Capture industry / tier / rate despite mixed [] and () wrappers.
TAG_PATTERN = re.compile(
    r'^\s*[\[\(\s]*?(?P<industry>.*?)\s*[\]\)]*\s*,\s*'
    r'[\[\(\s]*(?P<tier>[abc])[\]\)\s]*\s*,\s*'
    r'[\[\(\s]*take\s*rate\s*:\s*(?P<rate>\d+(?:\.\d+)?)',
    flags=re.IGNORECASE,
)

def parse_tag(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.Series({'industry': pd.NA, 'merchant_tier': pd.NA, 'take_rate': np.nan})
    text = str(value).strip()
    if not text or text.lower() in ('none', 'nan', '<na>'):
        return pd.Series({'industry': pd.NA, 'merchant_tier': pd.NA, 'take_rate': np.nan})
    match = TAG_PATTERN.search(text)
    if not match:
        return pd.Series({'industry': pd.NA, 'merchant_tier': pd.NA, 'take_rate': np.nan})
    industry = re.sub(r'\s+', ' ', match.group('industry').strip(' []()').lower())
    return pd.Series({
        'industry': industry if industry else pd.NA,
        'merchant_tier': match.group('tier').lower(),
        'take_rate': float(match.group('rate')),
    })

parsed = merchants['tags'].apply(parse_tag)
merchants = pd.concat([merchants, parsed], axis=1)
merchants['tag_parse_failed'] = merchants['tags'].notna() & merchants['industry'].isna()
print('Missing/uncategorisable industries:', merchants['industry'].isna().sum())
print('Non-null tags that failed parsing:', merchants['tag_parse_failed'].sum())
print('Distinct cleaned industries:', merchants['industry'].nunique())
display(merchants[['tags', 'industry', 'merchant_tier', 'take_rate']].head(12))
display(merchants['industry'].value_counts(dropna=False).head(40).rename_axis('industry').reset_index(name='merchants'))

## 3. Define five business-oriented segments

These are **analyst-defined groupings**, not categories supplied by the dataset. Review the unmatched industry table below and expand keywords if necessary. **Other** and **Unknown** are preserved, not silently dropped.

In [ ]:
# Ordered keyword rules: first match wins. Keep them interpretable and easy to adjust.
SEGMENT_KEYWORDS = {
    'Fashion & Lifestyle': [
        'shoe', 'clothing', 'apparel', 'fashion', 'jewel', 'watch', 'gift', 'novelty',
        'souvenir', 'florist', 'flower', 'beauty', 'cosmetic', 'accessor', 'leather',
        'luggage', 'department store', 'sporting goods', 'toy', 'game shop',
    ],
    'Electronics & Digital': [
        'digital goods', 'computer', 'software', 'telecom', 'telephone', 'electronic',
        'cable', 'satellite', 'television', 'radio services', 'camera', 'photographic',
        'internet', 'data processing', 'programming', 'appliance',
    ],
    'Home & Living': [
        'furniture', 'home furnishing', 'household', 'hardware', 'building material',
        'garden', 'nursery stock', 'floor covering', 'carpet', 'interior', 'lighting',
        'kitchen', 'home improvement', 'paint', 'decor',
    ],
    'Automotive & Outdoor': [
        'motor vehicle', 'automotive', 'automobile', 'auto part', 'bicycle', 'tent',
        'awning', 'camping', 'marine', 'boat', 'motorcycle', 'tyre', 'tire',
        'recreational vehicle', 'outdoor',
    ],
    'Arts, Media & Services': [
        'art dealer', 'art galler', 'book', 'periodical', 'newspaper', 'music',
        'instrument', 'sheet music', 'entertainment', 'cinema', 'theatre', 'theater',
        'professional service', 'repair', 'education', 'travel', 'hotel',
        'restaurant', 'cafe', 'food', 'health', 'medical', 'optician', 'optical',
    ],
}

SEGMENTS = list(SEGMENT_KEYWORDS)

def assign_segment(industry):
    if pd.isna(industry):
        return 'Unknown'
    for segment, keywords in SEGMENT_KEYWORDS.items():
        if any(keyword in industry for keyword in keywords):
            return segment
    return 'Other'

merchants['segment'] = merchants['industry'].apply(assign_segment)

segment_counts = merchants['segment'].value_counts().rename_axis('segment').reset_index(name='merchant_count')
segment_counts['share_pct'] = (100 * segment_counts['merchant_count'] / len(merchants)).round(1)
display(segment_counts)

print('Unmapped industries (inspect and add keywords if important):')
display(merchants.loc[merchants['segment'].eq('Other'), 'industry']
        .value_counts().rename_axis('industry').reset_index(name='merchant_count').head(60))

print('Top industries in each of the five segments:')
for segment in SEGMENTS:
    print(f'\n{segment}')
    display(merchants.loc[merchants['segment'].eq(segment), 'industry'].value_counts().head(12))

assert merchants['segment'].notna().all()
if any((merchants['segment'] == segment).sum() < 10 for segment in SEGMENTS):
    print('WARNING: At least one segment has fewer than 10 merchants. Review mapping before finalising.')

## 4. Rank merchants within each segment

**No new scoring weights:** Within each segment, sort by the existing `ranking_score`, which already includes predicted 3-month revenue and the fraud penalty. Ties use overall rank (if available), then merchant ABN. Keep flagged merchants in the results, clearly marked for review.

In [ ]:
if 'overall_rank' not in merchants.columns:
    merchants['overall_rank'] = merchants['ranking_score'].rank(method='first', ascending=False).astype(int)

sort_cols = ['segment', 'ranking_score', 'overall_rank', 'merchant_abn']
all_segment_rankings = merchants.sort_values(
    sort_cols, ascending=[True, False, True, True], kind='stable'
).copy()
all_segment_rankings['segment_rank'] = all_segment_rankings.groupby('segment').cumcount() + 1

top_10_per_segment = all_segment_rankings.loc[
    all_segment_rankings['segment'].isin(SEGMENTS) & (all_segment_rankings['segment_rank'] <= 10)
].copy()

show = ['segment', 'segment_rank', 'overall_rank', 'merchant_abn']
show += [c for c in ['name_merchant', 'industry', 'merchant_tier', 'ranking_score',
                     'predicted_3m_revenue', 'total_transaction_value',
                     'unique_customers', 'repeat_customer_share', 'take_rate',
                     'fraud_risk_flag', 'fraud_data_review_flag', 'manual_review_flag']
         if c in top_10_per_segment.columns]

for segment in SEGMENTS:
    print(f'\n### {segment} — Top 10')
    display(top_10_per_segment.loc[top_10_per_segment['segment'].eq(segment), show].reset_index(drop=True))

print('Selected merchants:', len(top_10_per_segment))

## 5. Segment comparison and risk review

Interpret figures as descriptive only. Review flagged merchants before making onboarding recommendations; the score does not automatically approve a merchant.

In [ ]:
summary = merchants.groupby('segment', dropna=False).agg(
    merchant_count=('merchant_abn', 'size'),
    median_ranking_score=('ranking_score', 'median'),
    mean_ranking_score=('ranking_score', 'mean'),
)
for col, label in [
    ('predicted_3m_revenue', 'median_predicted_3m_revenue'),
    ('total_transaction_value', 'median_historical_transaction_value'),
    ('unique_customers', 'median_unique_customers'),
    ('take_rate', 'median_take_rate'),
]:
    if col in merchants.columns:
        summary[label] = merchants.groupby('segment')[col].median()
for col in ['fraud_risk_flag', 'fraud_data_review_flag', 'manual_review_flag']:
    if col in merchants.columns:
        summary[col + '_share'] = merchants.groupby('segment')[col].mean()
summary = summary.reset_index().sort_values('merchant_count', ascending=False)
display(summary)

if 'manual_review_flag' in top_10_per_segment.columns:
    print('Top 10 merchants flagged for manual review by segment:')
    display(top_10_per_segment.groupby('segment')['manual_review_flag'].sum().rename('flagged_merchants'))

fig, ax = plt.subplots(figsize=(11, 5))
segment_counts.plot.bar(x='segment', y='merchant_count', ax=ax, legend=False)
ax.set(title='Merchant counts by segment', xlabel='', ylabel='Merchants')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
plot_df = top_10_per_segment.groupby('segment')['ranking_score'].mean().reindex(SEGMENTS)
plot_df.plot.bar(ax=ax)
ax.set(title='Mean ranking score of Top 10 merchants by segment', xlabel='', ylabel='Mean ranking score')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 6. Validate and export

Outputs are saved separately from the overall Top 100 ranking. The five selected segments should produce 50 rows if each has at least ten merchants.

In [ ]:
assert len(all_segment_rankings) == len(merchants)
assert all_segment_rankings['merchant_abn'].is_unique
assert top_10_per_segment.groupby('segment').size().le(10).all()
assert top_10_per_segment['segment_rank'].between(1, 10).all()
assert set(top_10_per_segment['segment']).issubset(SEGMENTS)
assert all_segment_rankings.groupby('segment')['segment_rank'].min().eq(1).all()

all_segment_rankings.to_parquet(OUTPUT / 'merchant_segment_rankings_all.parquet', index=False)
all_segment_rankings.to_csv(OUTPUT / 'merchant_segment_rankings_all.csv', index=False)
top_10_per_segment.to_parquet(OUTPUT / 'top_10_per_segment.parquet', index=False)
top_10_per_segment.to_csv(OUTPUT / 'top_10_per_segment.csv', index=False)
segment_counts.to_csv(OUTPUT / 'segment_counts.csv', index=False)
summary.to_csv(OUTPUT / 'segment_summary.csv', index=False)

print('Saved outputs to:', OUTPUT)
for path in sorted(OUTPUT.glob('*')):
    print(' -', path.name)
print('Top 10 counts:', top_10_per_segment.groupby('segment').size().to_dict())

## 7. Notes for the final report / presentation

- **Segmentation rationale:** Five industry-based segments provide interpretable comparisons and help assess portfolio diversification.
- **Ranking method:** Within each segment, retain the same overall weighted merchant score (including predicted 3-month revenue and fraud penalty) to ensure consistency with the overall Top 100.
- **Data limitations:** Merchant tags are messy and missing for some merchants. Unmapped labels are retained as `Other`, missing/unparseable tags as `Unknown`; audit these before finalising.
- **Risk:** Fraud and data-quality flags remain visible; a high score is not automatic approval.
- **Commercial interpretation:** `take_rate` is extracted but **not used in scoring**. If its units and relationship to BNPL fee revenue are confirmed, a separate take-rate-adjusted sensitivity analysis could be justified.
- **Next:** Review the top industries within each segment and adjust the keyword mapping if the current five groups are unbalanced or poorly matched.